In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LeakyReLU, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


In [ ]:
def build_generator(latent_dim):
    input = Input(shape=(latent_dim,))
    x = Dense(128)(input)
    x = LeakyReLU(alpha=0.2)(x)
    x = BatchNormalization()(x)
    x = Dense(256)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = BatchNormalization()(x)
    x = Dense(512)(x)
    x = LeakyReLU(alpha=0.2)(x)
    output = Dense(units=data_dim, activation='tanh')(x)
    generator = Model(inputs=input, outputs=output)
    return generator


In [ ]:
def build_discriminator(data_dim):
    input = Input(shape=(data_dim,))
    x = Dense(512)(input)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(256)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(128)(x)
    x = LeakyReLU(alpha=0.2)(x)
    output = Dense(1, activation='sigmoid')(x)
    discriminator = Model(inputs=input, outputs=output)
    discriminator.compile(optimizer=Adam(0.0002, 0.5), loss='binary_crossentropy', metrics=['accuracy'])
    return discriminator


In [ ]:
def build_gan(generator, discriminator):
    discriminator.trainable = False  # Freeze the discriminator during the generator training
    gan_input = Input(shape=(latent_dim,))
    generated_data = generator(gan_input)
    gan_output = discriminator(generated_data)
    gan = Model(gan_input, gan_output)
    gan.compile(optimizer=Adam(0.0002, 0.5), loss='binary_crossentropy')
    return gan


In [ ]:
def train_gan(generator, discriminator, gan, data, epochs, batch_size, latent_dim):
    valid = np.ones((batch_size, 1))  # Labels for real data
    fake = np.zeros((batch_size, 1))  # Labels for fake data
    
    for epoch in range(epochs):
        # Train discriminator
        idx = np.random.randint(0, data.shape[0], batch_size)
        real_data = data[idx]
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        generated_data = generator.predict(noise)
        
        d_loss_real = discriminator.train_on_batch(real_data, valid)
        d_loss_fake = discriminator.train_on_batch(generated_data, fake)
        d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
        
        # Train generator
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        g_loss = gan.train_on_batch(noise, valid)
        
        # Optionally print progress and save models
        print(f"Epoch {epoch + 1} / {epochs}, Discriminator Loss: {d_loss[0]}, Generator Loss: {g_loss}")
